In [ ]:
# 必要パッケージの読み込み
if (!require("mice")) install.packages("mice")
if (!require("ggplot2")) install.packages("ggplot2")
if (!require("stringr")) install.packages("stringr")
if (!require("parallel")) install.packages("parallel")

library(mice)
library(ggplot2)
library(stringr)
library(parallel)

# --- 1. 設定 -----------------------------------------------------------------
# 入力データのパス
input_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/MICE_check"

# 出力フォルダのパス (ご指定のフォルダ)
output_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/MICE_check/Untitled Folder/Untitled Folder"

# フォルダが存在しない場合は作成
if (!dir.exists(output_path)) dir.create(output_path, recursive = TRUE)

setwd(input_path)

files <- c(
  "Data20211217Final.csvrfN4maxit5R0.9893.csv",
  "Data20211217Final.csvcartN10maxit4R0.9791new.csv",
  "Data20211217Final.csvpmmN8maxit7R0.8281.csv"
)

n_trials <- 5         # 10回から5回に削減
n_mask_cells <- 50
n_cores <- 10         # 63コアから10コアに調整 (メモリ状況を見て増やしてください)

# --- 2. パラメータ抽出関数 ---------------------------------------------------
get_params_from_name <- function(fname) {
  method <- ifelse(str_detect(fname, "rf"), "rf",
            ifelse(str_detect(fname, "cart"), "cart",
            ifelse(str_detect(fname, "pmm"), "pmm", "")))
  m_val <- as.numeric(str_extract(fname, "(?<=N)\\d+"))
  maxit_val <- as.numeric(str_extract(fname, "(?<=maxit)\\d+"))
  return(list(method = method, m = m_val, maxit = maxit_val))
}

# --- 3. 試行実行関数 ---------------------------------------------------------
run_single_trial <- function(trial_idx, df_orig, params, num_col_names) {
  library(mice)
  
  num_cols_data <- as.matrix(df_orig[, num_col_names])
  obs_idx <- which(!is.na(num_cols_data), arr.ind = TRUE)
  
  # シード値を42ベースで固定
  set.seed(42 + trial_idx)
  
  mask_sel <- obs_idx[sample(nrow(obs_idx), 50), ]
  
  df_masked <- df_orig
  ground_truth <- numeric(50)
  
  for(i in 1:50) {
    r <- mask_sel[i, 1]; c_idx <- mask_sel[i, 2]
    c_name <- num_col_names[c_idx]
    ground_truth[i] <- df_orig[r, c_name]
    df_masked[r, c_name] <- NA
  }
  
  # MICE実行
  imp <- mice(df_masked, m = params$m, maxit = params$maxit, 
              method = params$method, printFlag = FALSE)
  df_complete <- complete(imp)
  
  # 誤差計算
  imputed_vals <- numeric(50)
  for(i in 1:50) {
    r <- mask_sel[i, 1]; c_name <- num_col_names[mask_sel[i, 2]]
    imputed_vals[i] <- df_complete[r, c_name]
  }
  
  rmse <- sqrt(mean((ground_truth - imputed_vals)^2, na.rm = TRUE))
  mae  <- mean(abs(ground_truth - imputed_vals), na.rm = TRUE)
  cor_val <- cor(ground_truth, imputed_vals, use = "complete.obs")
  
  return(data.frame(Trial = trial_idx, RMSE = rmse, MAE = mae, Correlation = cor_val))
}

# --- 4. メイン処理 -----------------------------------------------------------
all_results <- data.frame()

total_files <- length(files)
cat("\n=== MICE Accuracy Validation (Parallel) ===\n")
cat("Output to:", output_path, "\n")
cat("Parameters: n_trials=5, Seed=42, Cores=", n_cores, "\n\n")

for (i in seq_along(files)) {
  f <- files[i]
  if (!file.exists(f)) next
  
  params <- get_params_from_name(f)
  
  # 現在の進捗を表示
  timestamp <- format(Sys.time(), "%H:%M:%S")
  cat(sprintf("[%s] %d/%d: Processing %s (Method: %s)...\n", 
              timestamp, i, total_files, f, params$method))
  
  df_orig <- read.csv(f, row.names = 1, stringsAsFactors = TRUE)
  num_cols <- sapply(df_orig, is.numeric)
  num_col_names <- names(df_orig)[num_cols]

  # 並列処理実行
  trial_results_list <- mclapply(
    1:n_trials, 
    run_single_trial, 
    df_orig = df_orig, 
    params = params, 
    num_col_names = num_col_names, 
    mc.cores = n_cores,
    mc.set.seed = TRUE
  )
  
  trial_results_df <- do.call(rbind, trial_results_list)
  trial_results_df$Method <- params$method
  trial_results_df$FileName <- f
  
  all_results <- rbind(all_results, trial_results_df)
  
  cat(sprintf("   -> [OK] Finished %s.\n", f))
}

# --- 5. 保存と可視化 ---------------------------------------------------------
cat("\nFinalizing plots and tables...\n")

# 出力先をUntitled Folder/Untitled Folderへ変更
setwd(output_path)

# 統計サマリー
summary_stats <- aggregate(cbind(RMSE, MAE, Correlation) ~ Method, data = all_results, FUN = mean)
write.csv(all_results, "MICE_Trial_Full_Results_v2.csv", row.names = FALSE)
write.csv(summary_stats, "MICE_Trial_Summary_Average_v2.csv", row.names = FALSE)

# ボックスプロット
p <- ggplot(all_results, aes(x = Method, y = RMSE, fill = Method)) +
  geom_boxplot(alpha = 0.7) +
  geom_jitter(width = 0.1, size = 2) +
  theme_minimal(base_size = 15) +
  labs(title = "MICE Imputation Accuracy (5 Trials, Seed=42)",
       subtitle = "Exactly 50 cells masked per trial",
       y = "RMSE (Lower is better)",
       x = "Method (Parsed from Filename)")

ggsave("MICE_Parallel_Accuracy_Boxplot_v2.png", plot = p, width = 10, height = 6)

cat("\n=== ALL PROCESS COMPLETE ===\n")
cat("Results are located in:\n", output_path, "\n")

Loading required package: mice


Attaching package: 'mice'


The following object is masked from 'package:stats':

    filter


The following objects are masked from 'package:base':

    cbind, rbind


Loading required package: ggplot2

Loading required package: stringr

Loading required package: parallel




=== MICE Accuracy Validation (Parallel) ===
Output to: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/MICE_check/Untitled Folder/Untitled Folder 
Parameters: n_trials=5, Seed=42, Cores= 10 

